<a href="https://colab.research.google.com/github/jeffheaton/app_generative_ai/blob/main/t81_559_class_05_1_langchain_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T81-559: Applications of Generative Artificial Intelligence
**Module 5: LangChain: Data Extraction**
* Instructor: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.wustl.edu/Programs/Pages/default.aspx)
* For more information visit the [class website](https://github.com/jeffheaton/app_generative_ai).

# Module 5 Material

* **Part 5.1: The Structured Output Problem** [[Video]](https://www.youtube.com/watch?v=62CSR141VRE) [[Notebook]](t81_559_class_05_1_langchain_data.ipynb)
* Part 5.2: Designing Schemas with Pydantic [[Video]](https://www.youtube.com/watch?v=VXm8gPzU3qc) [[Notebook]](t81_559_class_05_2_parsers.ipynb)
* Part 5.3: Validation, Retries, and Refusals [[Video]](https://www.youtube.com/watch?v=dc4fn-W60hg) [[Notebook]](t81_559_class_05_3_pydantic.ipynb)
* Part 5.4: Structured Extraction at Scale [[Video]](https://www.youtube.com/watch?v=jBpkAblQC_U) [[Notebook]](t81_559_class_05_4_custom_parsers.ipynb)
* Part 5.5: Structured Output Under the Hood [[Video]](https://www.youtube.com/watch?v=_txWiLjf4bo) [[Notebook]](t81_559_class_05_5_output_fixing_parsers.ipynb)

# Google CoLab Instructions

The following code ensures that Google CoLab is running and maps Google Drive if needed.

In [6]:
import os

try:
    from google.colab import drive, userdata
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# OpenAI Secrets
if COLAB:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Install needed libraries in CoLab
if COLAB:
    !pip install langchain langchain_openai

Note: using Google CoLab


# 5.1: The Structured Output Problem

Large language models produce prose, but programs need data. Nearly everything you will build after this module -- retrieval pipelines, agents, dashboards, batch jobs -- must take what a model says and hand it to code that expects concrete values: a number, a date, a list, a table row. An essay, however eloquent, is useless to a DataFrame.

In the early days of LLM applications, developers solved this by begging the model for a format ("respond ONLY with comma-separated values...") and then parsing the returned text with string operations. Entire libraries of "output parsers" grew up around this workaround, and earlier versions of this course taught them. Modern models and APIs have made most of that machinery obsolete: today you declare a *schema* -- a typed description of the data you want -- and the API constrains the model to produce output matching it. You receive a validated Python object, not a string to clean up.

This module covers structured output from first principles to production practice:

* **Part 5.1**: The problem, and the one-call modern solution.
* **Part 5.2**: Designing schemas with Pydantic -- the craft that replaces prompt-format hacks.
* **Part 5.3**: Validation, retries, and refusals -- what the schema guarantees and what it cannot.
* **Part 5.4**: Extraction at scale -- turning a pile of documents into a dataset.
* **Part 5.5**: How structured output works under the hood, and what to do when it is not available.

## The Fragile Way: Parsing Prose

To appreciate the modern approach, we first look at the problem it solves. The following code asks the model a simple question and prints the free-text answer.

In [7]:
from langchain_openai import ChatOpenAI

MODEL = 'gpt-5.6-luna'

llm = ChatOpenAI(model=MODEL)

response = llm.invoke(
    "Recommend one classic science fiction movie. "
    "Give the title, the year it was released, and a list of its genres."
)
print(response.content)

**Blade Runner** — **1982**

**Genres:**
- Science fiction
- Neo-noir
- Dystopian
- Thriller
- Drama


The answer is correct and readable, but look at it as a programmer rather than a reader. Which line holds the title? Is the year in parentheses, after a colon, or embedded in a sentence? Are the genres comma-separated, bulleted, or written as prose? Run the cell several times and you will see the formatting drift, because nothing about the request pins it down.

The traditional fix was code like the following, which works only for the exact formatting the model happened to use on the day you wrote it.

In [8]:
# A naive parser: this depends entirely on the model's formatting mood.
def parse_movie(text):
    title, year, genres = None, None, []
    for line in text.splitlines():
        low = line.lower()
        if "title" in low and ":" in line:
            title = line.split(":", 1)[1].strip()
        elif "year" in low and ":" in line:
            year = line.split(":", 1)[1].strip()
        elif "genre" in low and ":" in line:
            genres = [g.strip() for g in line.split(":", 1)[1].split(",")]
    return title, year, genres

print(parse_movie(response.content))

(None, None, ['**'])


Sometimes that returns three useful values; sometimes it returns `(None, None, [])` because the model wrote a paragraph instead of labeled lines. Building a business process on this is how LLM projects used to fail.

## The Modern Way: Declare a Schema

[Pydantic](https://docs.pydantic.dev/) is Python's standard library for typed data models. A Pydantic class *is* a schema: its fields declare names and types, and each field can carry a description. LangChain's `with_structured_output` method binds such a schema to a chat model. The API then constrains the model's generation so that the response must match the schema, and LangChain hands you back a validated instance of your class.

In [9]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie recommendation."""
    title: str
    year: int = Field(description="Year of first theatrical release")
    genres: list[str]

structured_llm = llm.with_structured_output(Movie)

movie = structured_llm.invoke(
    "Recommend one classic science fiction movie."
)

print(movie)
print()
print(f"Title:  {movie.title}")
print(f"Year:   {movie.year + 0}  (a real int -- arithmetic works)")
print(f"Genres: {movie.genres}")
print(f"Type:   {type(movie)}")

title='2001: A Space Odyssey' year=1968 genres=['Science Fiction', 'Drama']

Title:  2001: A Space Odyssey
Year:   1968  (a real int -- arithmetic works)
Genres: ['Science Fiction', 'Drama']
Type:   <class '__main__.Movie'>


Notice everything that is *absent* from that code: there are no format instructions in the prompt, no parsing, no string cleanup, and no possibility of a missing field. The prompt states the task; the schema states the shape. `movie.year` is an `int` you can compare and compute with, and `movie.genres` is a real Python list.

The docstring on the class and the `description` on each field are not decoration -- they are transmitted to the model as part of the schema and guide what it fills in. Writing good descriptions is the new prompt engineering, and it is the subject of Part 5.2.

## Returning Collections

A schema can contain lists of other models, which is how you ask for several records in one call.

In [10]:
class MovieList(BaseModel):
    """A list of movie recommendations."""
    movies: list[Movie]

structured_llm = llm.with_structured_output(MovieList)

result = structured_llm.invoke(
    "Recommend three classic science fiction movies from three different decades."
)

for m in result.movies:
    print(f"{m.year}  {m.title}  ({', '.join(m.genres)})")

1968  2001: A Space Odyssey  (Science Fiction, Drama)
1982  Blade Runner  (Science Fiction, Action)
1999  The Matrix  (Science Fiction, Action)


## Why This Matters for the Rest of the Course

Structured output is not a convenience feature; it is the connective tissue of applied generative AI, and you will meet it again in nearly every remaining module:

* In **Module 6 (RAG)**, structured metadata makes retrieved documents filterable.
* In **Module 7 (Agents)**, every tool an agent can call declares its arguments as exactly this kind of Pydantic schema -- an agent calling a tool *is* structured output.
* In **Part 5.4**, we will use it to turn five hundred unstructured documents into a clean dataset, which is arguably the single most common production use of LLMs today.

Before we can do that responsibly, we need two skills: designing schemas well (Part 5.2) and knowing what a validated object does -- and does not -- guarantee (Part 5.3).